In [1]:
import torch
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.transforms import v2
from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights,densenet121,DenseNet121_Weights
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import random
import os
import timm
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, ConfusionMatrixDisplay
from copy import deepcopy
from typing import Tuple, List, Dict
import csv
import cv2
from sklearn.manifold import TSNE
import pandas as pd
import seaborn as sns
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GradCAM(torch.nn.Module):
    def __init__(self,model,target_layer):
        super(GradCAM, self).__init__()
        
        # get the pretrained VGG19 network
        self.model = model.eval()
        self.target_layer = target_layer
        
        self.activation = None
        self.gradient = None

        target_layer.register_forward_hook(self.hook_activation)
        target_layer.register_forward_hook(self.hook_gradient)
    
    def hook_activation(self, module, input, output):
        self.activation = output.cpu().detach()

    def hook_gradient(self, module, input,output):
        def save_grad(grad):
            self.gradient = grad.cpu().detach()
        output.register_hook(save_grad)

    def __call__(self, x):
        self.activation = None
        self.gradients = None
        return self.model(x)
    
    # method for the gradient extraction
    def get_activation_gradient(self):
        return self.activation, self.gradient
    
def save_gradcam(model, sample, image, class_int, img_name,image_shape):
    output=model(sample)

    output[:,class_int].backward()

    activation,gradient = model.get_activation_gradient()
    activation,gradient = activation.squeeze(0), gradient.squeeze(0)


    gradient = torch.mean(gradient,dim=[1,2])
    activation=activation*gradient.reshape(-1,1,1)

    heatmap = activation.mean(dim=0)
    heatmap = np.maximum(heatmap, 0)
    heatmap /= torch.max(heatmap)
    heatmap = heatmap.cpu().detach().data.numpy()
    heatmap=cv2.resize(heatmap,(image_shape,image_shape))
    heatmap = cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET)

    cv2.imwrite(img_name,image+0.5*heatmap)

class KONet(torch.nn.Module):

    def __init__(
            self,
            m1_ratio=0.4,
            m2_ratio=0.6,
            m1_dropout=0.1,
            m2_dropout=0.3,
            n_classes=2
    ):
        super().__init__()
        assert m1_ratio+m2_ratio==1
        self.n_classes=n_classes
        self.m1_ratio=m1_ratio #EfficientNet
        self.m2_ratio=m2_ratio #DenseNet
        self.m1_dropout=m1_dropout
        self.m2_dropout=m2_dropout

        self.efficient=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.efficient.classifier[0]=torch.nn.Dropout(p=self.m1_dropout,inplace=True)
        self.efficient.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=self.n_classes)

        self.dense=densenet121(weights=DenseNet121_Weights.DEFAULT)
        self.dense.classifier=torch.nn.Sequential(torch.nn.Dropout(p=self.m2_dropout,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )

    def forward(self, x):
        m1=self.efficient(x)
        m2=self.dense(x)
        out=self.m1_ratio*m1+self.m2_ratio*m2
        return out

def ewc_init(ewc_model,dataloader,loss_fn,optimizer):
    ewc_model.train()
    optimizer.zero_grad()

    optpar_dict={}
    fisher_dict={}
    for name, param in ewc_model.named_parameters():
        fisher_dict[name] = torch.zeros_like(param)

    for train_data,train_label in tqdm(dataloader):
        train_data , train_label=train_data.to(device) , train_label.to(device)
        output=ewc_model(train_data)

        loss=loss_fn(output , train_label)
        loss.backward()

        for name, param in ewc_model.named_parameters():
            if param.grad is not None:
                fisher_dict[name] += (param.grad.detach() ** 2)

    #After iterating through dataset, save the parameters and the gradients squared in a dict
    for name , param in ewc_model.named_parameters():
        #if "classifier" in name:
        optpar_dict[name] = deepcopy(param)
        optpar_dict[name].requires_grad=False

        #fisher_dict[name] = deepcopy(param.grad).pow(2) / len(dataloader)
        #fisher_dict[name].requires_grad=False
    for name in fisher_dict:
        fisher_dict[name] /= len(dataloader)
    return ewc_model,optpar_dict,fisher_dict     

def ewc_loss(ewc_model,optpar_dict,fisher_dict,ewc_lambda=8):
    distill_loss=0
    for name , param in ewc_model.named_parameters():
        #if "classifier" in name:
        optpar = optpar_dict[name]
        fisher = fisher_dict[name]

        distill_loss+= (fisher * (optpar - param).pow(2)).sum() * ewc_lambda
    return distill_loss
    
def test(model,dataloader,loss_fn):
    model.eval()
    loss=0
    labels=[]
    probabilities=[]
    for data,label in dataloader:
        with torch.no_grad():
            data , label=data.to(device) , label.to(device)

            output=model(data)
            loss+=loss_fn(output , label)
            prob=output.softmax(dim=1)
            labels.append(label.detach().cpu().numpy())
            probabilities.append(prob.detach().cpu().numpy())

    labels=np.concatenate(labels,axis=0)
    probabilities=np.concatenate(probabilities,axis=0)

    loss=loss/len(dataloader)
    return loss.item(),labels,probabilities


def set_random_seed(seed: int = 2222, deterministic: bool = False):
        random.seed(seed)
        np.random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)  # type: ignore
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = deterministic  # type: ignore

class CustomImageFolder(torchvision.datasets.ImageFolder):
    def __init__(
        self,
        root,
        transform= None,
        class_map = None
    ):
        self.map = class_map
        super().__init__(
            root,
            transform=transform,
        )
        

    def find_classes(self, directory: str) -> Tuple[List[str], Dict[str, int]]:
        """
        Override this method to load from setting file instead of scanning directory
        """
        classes = list(self.map.keys())
        classes_to_idx = self.map
        return classes, classes_to_idx
def prep_dataset(path, image_shape=224, augmented_dataset_size=1800,class_map=None,
                 train_split=0.8, valid_split=0.1, test_split=0.1, seed=42):

    non_augment_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])

    aug_transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32,scale=True),
        v2.RandomAffine(degrees=30, shear=30),
        v2.RandomZoomOut(side_range=(1, 1.5)),
        v2.Resize((image_shape, image_shape), antialias=True),
        v2.Normalize(mean=[0.5], std=[0.5]),
    ])
    base_dataset = CustomImageFolder(path, transform=non_augment_transform,class_map=class_map)
    augmented_dataset = CustomImageFolder(path, transform=aug_transform,class_map=class_map)
    n_samples = len(base_dataset)

    labels = [s[1] for s in base_dataset.samples]
    indices = np.arange(n_samples)

    # split test first
    train_val_idx, test_idx = train_test_split(indices, test_size=test_split, stratify=labels, random_state=seed)

    kf = KFold(n_splits=10, shuffle=True, random_state=seed)
    splits = list(kf.split(train_val_idx))

    fold_datasets=[]
    factor=augmented_dataset_size//len(base_dataset)
    
    for fold, (train_indices, val_indices) in enumerate(splits):
        augmented_train_indices = np.repeat(train_indices, factor)
        train_set = torch.utils.data.Subset(augmented_dataset, augmented_train_indices)
        valid_set = torch.utils.data.Subset(base_dataset, val_indices)
        fold_datasets.append((fold+1,train_set,valid_set))

    test_subset = torch.utils.data.Subset(base_dataset, test_idx)


    return fold_datasets, test_subset

def distillation_loss(new_logits,old_logits,T=2):
    outputs=new_logits
	#outputs = torch.log_softmax(new_logits/T, dim=1)   # compute the log of softmax values
    labels = torch.softmax(old_logits/T, dim=1)
    # print('outputs: ', outputs)
    # print('labels: ', labels.shape)
    outputs = torch.sum(outputs * labels, dim=1, keepdim=False)
    outputs = -torch.mean(outputs, dim=0, keepdim=False)
    # print('OUT: ', outputs)
    return outputs

In [2]:
n_classes=3
image_shape=224
augmented_dataset_size=1800
batch_size=32
epochs=20
#ewc_lambda=2
lwf_lambda=1.5
T=2
seed=42
model_name= 'dense'
script_name = 'train osteopenia incremental lwf&ewc'
results_file = "fold_results.csv"
path1="/kaggle/input/datasets/haiderazam/kod-2-class/Osteoporosis Knee X-ray Preprocessed"
path2="/kaggle/input/datasets/haiderazam/kod-only-osteopenia/Osteoporosis Knee X-ray only osteopenia Preprocessed"
weigths_path="/kaggle/input/datasets/haiderazam/osteopenia-incremental-lwf-ewc-weights/train osteopenia incremental lwf_ewc/model"
os.makedirs('model',exist_ok=True)

In [3]:
map1={'normal':0,'osteoporosis':2}
map2={'osteopenia':1}

set_random_seed(seed)

fold_datasets1,test_set1=prep_dataset(path1,image_shape,augmented_dataset_size,map1)
fold_datasets2,test_set2=prep_dataset(path2,image_shape,augmented_dataset_size,map2)

best_acc=0
best_loss=np.inf
print('Model: ',model_name)
for (fold,train_set1,valid_set1),(_,train_set2,valid_set2) in zip(fold_datasets1,fold_datasets2):
    print(f"Fold {fold}")

    print(f"Test set 1 size: {len(test_set1)}, Test set 2 size: {len(test_set2)}")

    test_set = torch.utils.data.ConcatDataset([test_set1, test_set2])
    
    test_dataloader = DataLoader(test_set, batch_size=batch_size, num_workers=4, pin_memory=True,
                                    persistent_workers=True, shuffle=False)
    
    if 'efficient' in model_name:
        model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        p=0.1
        model.classifier[0]=torch.nn.Dropout(p=p,inplace=True)
        model.classifier[-1]=torch.nn.Linear(in_features=1280,out_features=n_classes)
    
    elif 'dense' in model_name:
        model=densenet121(weights=DenseNet121_Weights.DEFAULT)
        p=0.3
        model.classifier=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=1024,out_features=n_classes),
                                            )
    
    elif 'conv_next' in model_name:
        p=0.3
        model=torchvision.models.convnext_tiny(weights='DEFAULT')
        model.classifier[2]=torch.nn.Sequential(torch.nn.Dropout(p=p,inplace=True),
                                            torch.nn.Linear(in_features=768,out_features=n_classes),
                                            )
    elif 'edgenext' in model_name:
        model=timm.create_model("edgenext_small.usi_in1k",
                                        pretrained=True, 
                                        features_only=False,
                                        in_chans=3,
                                        num_classes=n_classes,
                                        global_pool='avg'
                                        )
    
    elif 'KONet' in model_name:
        m1_ratio=0.6
        m2_ratio=0.4
        m1_dropout=0.1
        m2_dropout=0.3
        model=KONet(m1_ratio=m1_ratio,m2_ratio=m2_ratio,m1_dropout=m1_dropout,m2_dropout=m2_dropout,n_classes=n_classes)
    
    elif 'mobilenet' in model_name:
        model=torchvision.models.mobilenet_v3_small(weights='DEFAULT')
        model.classifier[3]=torch.nn.Linear(in_features=1024,out_features=n_classes)

    
    model.load_state_dict(torch.load(f'{weigths_path}/{model_name}_fold_{fold}_best_param.pkl'))

    
    model.to(device)
    #optimizer=torch.optim.AdamW(model.parameters(),lr=0.0000625)
    optimizer=torch.optim.SGD(model.parameters(),lr=0.000008,momentum=0.9, weight_decay=5e-4)
    
    loss_fn=torch.nn.CrossEntropyLoss()

    model.eval()
    test_loss1, test_labels, test_probabilities = test(model, test_dataloader, loss_fn)
    test_pred_labels = np.argmax(test_probabilities, axis=1)
    test_acc1 = balanced_accuracy_score(test_labels, test_pred_labels)
    test_auc=roc_auc_score(test_labels,test_probabilities,average='weighted',multi_class='ovr')
    test_f1=f1_score(test_labels,test_pred_labels,average='weighted')
    if best_acc<test_acc1:
        best_acc=test_acc1
        best_fold=fold
        print(f"\nBest accuracy at fold {fold}")
    if best_loss>test_loss1:
        best_loss=test_loss1
        print(f"\nBest loss at fold {fold}")
    print(f"Combined Test set accuracy: {test_acc1}")
    print(f"F1-Score: {test_f1}")
    print(f"ROC_AUC: {test_auc}")


Model:  dense
Fold 1
Test set 1 size: 14, Test set 2 size: 20
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 171MB/s]



Best accuracy at fold 1

Best loss at fold 1
Combined Test set accuracy: 0.8666666666666667
F1-Score: 0.852187028657617
ROC_AUC: 0.9079670329670328
Fold 2
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accuracy: 0.7555555555555555
F1-Score: 0.793687230989957
ROC_AUC: 0.8123787976729152
Fold 3
Test set 1 size: 14, Test set 2 size: 20

Best loss at fold 3
Combined Test set accuracy: 0.7472222222222222
F1-Score: 0.8173591114767585
ROC_AUC: 0.964770523594053
Fold 4
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accuracy: 0.37222222222222223
F1-Score: 0.4739819004524886
ROC_AUC: 0.7110536522301228
Fold 5
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accuracy: 0.7944444444444444
F1-Score: 0.7958033892289947
ROC_AUC: 0.8355688429217841
Fold 6
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accuracy: 0.7194444444444444
F1-Score: 0.772110242698478
ROC_AUC: 0.8362960568842922
Fold 7
Test set 1 size: 14, Test set 2 size: 20
Combined Test set accura

In [4]:
model.load_state_dict(torch.load(f'{weigths_path}/{model_name}_fold_{best_fold}_best_param.pkl'))

def image_normalizer_misclassify(sample):
    image=(sample+1)*127
    image=image.to(torch.int64).cpu().numpy()
    image=image.transpose(1,2,0)
    return image

wrong_indices = []
model.eval()
os.makedirs('misclassify', exist_ok=True)
with torch.no_grad():
    for idx in range(len(test_set)):
        sample, label = test_set.__getitem__(idx)
        image = image_normalizer_misclassify(sample)
        sample = sample.unsqueeze(0).to(device)
        output = model(sample)
        pred = output.argmax(dim=1).item()
        if pred != label:
            wrong_indices.append((idx, pred, label))
            cv2.imwrite(f'misclassify/{idx},{pred},{label}.jpg',image)
print("Misclassified indices:", wrong_indices)

[ WARN:0@16.795] global loadsave.cpp:1063 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


Misclassified indices: [(4, 1, 2), (11, 1, 2), (23, 0, 1), (27, 2, 1), (29, 2, 1)]


In [5]:
map1={'normal':0,'osteoporosis':2}
map2={'osteopenia':1}
new_map={**map1, **map2}
idx_to_class = {v: k for k, v in new_map.items()}

os.makedirs('misclassify heatmaps', exist_ok=True)

#GradCAM part
if model_name=='efficient':
    target_layer = model.features[8][0]
    
elif model_name=='dense':
    target_layer = model.features[-2].denselayer16.conv2

elif 'mobilenet' in model_name:
    target_layer = model.features[-1][0]

elif 'edgenext' in model_name:
    target_layer = model.stages[-1].blocks[-1].convs[0]
    
gradcam_model=GradCAM(model,target_layer)
selected_samples = [(11, 1, 2), (27, 2, 1), (23, 0, 1)]
for idx, pred_class, true_class in selected_samples:
    sample=test_set.__getitem__(idx)[0].unsqueeze(0)
    sample = sample.to(device)

    image=image_normalizer_misclassify(sample[0])

    cv2.imwrite(f'misclassify heatmaps/Idx {idx} Predicted {idx_to_class[pred_class]} Actual {idx_to_class[true_class]}.jpg',image)

    img_name = f'misclassify heatmaps/Idx {idx} Predicted {idx_to_class[pred_class]} Actual {idx_to_class[true_class]} pred heatmap.jpg'
    save_gradcam(gradcam_model,sample,image,pred_class,img_name,image_shape)

    img_name = f'misclassify heatmaps/Idx {idx} Predicted {idx_to_class[pred_class]} Actual {idx_to_class[true_class]} actual heatmap.jpg'
    save_gradcam(gradcam_model,sample,image,true_class,img_name,image_shape)